# F02 학습모듈의 개요 ①·②·⑤·⑥·⑧ 검수 — v1.0

입력: 원고 1개 + 선택적인 개발계획서 1개 (`.hwp` / `.hwpx`, 순서 무관, 내용으로 역할 판별).
출력: 항목별 JSON, 요약, 지적사항, 전체 결과, 진단 정보, ZIP.

**③ 선수학습 코드 병기 · ④ 선수학습 능력단위 수준 · ⑦ 통합 개발 시 요소 전부 기재는 미구현입니다.**
이 결과만으로 개요 8개 항목 전체의 적합 여부를 판단하지 마세요.

### 판정
- ① 목표(형식 항목): `~할 수 있다.` 종결(O/X), 능력단위 정의의 주어(`…란`)가 남아 있지 않은가(O/X), 정의 내용과 대응하는가(공백·구두점을 뺀 유사도 0.85 이상이면 O, 낮으면 △ 사람 확인). 계획서 문장과 글자가 같은지는 묻지 않으며 계획서 '학습모듈의 목표' 줄은 참고로만 적습니다.
- ② 선수학습: 쉼표로 나눈 항목 수가 5개 이내인가만 봅니다(괄호 안 쉼표는 나누지 않음). '구체적인'은 사람 판단.
- ⑤ 내용 체계 표의 학습명·학습 내용명 ↔ 개발계획서(difflib로 누락/추가/변경/순서 구분). 차례 면과의 일치는 차례 검수 ③을 참조.
- ⑥ 요소 코드 번호 집합·코드별 요소 명칭 ↔ 개발계획서, 코드가 이 모듈의 능력단위 코드인가.
- ⑧ 핵심 용어 10개 이내(O/X). 명사형 여부는 종결어·길이 휴리스틱으로 △만 냅니다.
- 개발계획서가 없으면 ①의 대조는 생략(-), ⑤⑥은 △입니다.
- 글자 비교는 엄격합니다. 공백 차이도 다른 글자입니다(`객실·부대시설` ≠ `객실 · 부대시설`, `예약현황` ≠ `예약 현황`). 머리 번호(`1.` / `1-1.`)만 지우고 비교하며, 기호 글꼴 문자는 가운뎃점으로 바꿔 비교한 뒤 참고 비고로 적습니다(문자 자체는 차례 ①에서 X로 보고).

### 처리 흐름
파일 읽기 → 개요(목표/선수학습/내용 체계 표/핵심 용어)·계획서 모듈 블록 추출 → ①②⑤⑥⑧ 판정 → JSON/ZIP 저장.
기본 양식은 section0=차례·개요입니다. 아래 설정에서 변경할 수 있습니다.

위에서 아래로 실행하세요. Python 3.10 이상이며 추가 설치 패키지는 없습니다.
이 노트북은 `tools/build_notebook.py`로 Python 패키지에서 생성한 배포본입니다.
팀의 수정 원본은 `src/ncs_core/`(공통, 차례 검수와 같은 파일)와 `src/ncs_overview/`이고, 수정 후 노트북을 다시 생성합니다.

- `O`: 충족, `X`: 위반, `△`: 사람 확인, `-`: 해당 없음 또는 대조 생략.
- 행 판정(검수결과): X가 하나라도 있으면 `부적합`, X 없이 △가 있으면 `판단불가`, 나머지는 `적합`.
- `*_00_요약.json`의 `검수결과`는 검수 확인서 표현대로 적합/부적합/판단불가로 종합한다(부적합 > 판단불가 > 적합).


## 1. 실행 환경 준비
임시 폴더에 이 배포본의 패키지를 구성합니다. 기존 프로젝트 파일은 덮어쓰지 않습니다.


In [ ]:
from pathlib import Path
import importlib
import json
import sys
import tempfile

RUNTIME_DIR = Path(tempfile.mkdtemp(prefix="ncs_overview_"))
# 실패하거나 재실행해도 이전 결과를 다시 쓰지 않도록 초기화합니다.
result = saved = None
PATHS = []


def write_module(relative_path, source):
    path = RUNTIME_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")


print("검수 엔진 위치:", RUNTIME_DIR)


## 2. 검수 엔진
각 셀은 한 모듈의 소스입니다(접혀 있음). 사용만 할 때는 그대로 실행하세요.


In [ ]:
# 모듈: ncs_core/__init__.py
write_module('ncs_core/__init__.py', r'''"""F02 검수 공통 코어 (Python 3.10+, 추가 설치 패키지 없음).

두 검수 패키지(ncs_toc, ncs_position)가 공유하는 부분:
문서 읽기, 문자 정규화·판정, 입력 역할 판별, 모듈 코드, 규칙 실행, 결과 저장.
검수 정책(무엇을 X로 볼지)은 여기에 두지 않는다.
"""

from .documents import Document, Table, load_document
from .inputs import detect_role, select_documents, validate_distinct
from .models import ReviewResult, SavedReport
from .module_code import resolve_module_code, validate_module_code
from .reporting import save_report
from .rules import Rule, run_rules
from .text import FAIL, NA, O, PASS, PUA, TRI, UNDECIDED, VERDICTS, X, flatten, make_normalizer, verdict, visible

__all__ = [
    "Document",
    "Table",
    "load_document",
    "detect_role",
    "select_documents",
    "validate_distinct",
    "ReviewResult",
    "SavedReport",
    "resolve_module_code",
    "validate_module_code",
    "save_report",
    "Rule",
    "run_rules",
    "NA",
    "PASS",
    "FAIL",
    "UNDECIDED",
    "VERDICTS",
    "O",
    "PUA",
    "TRI",
    "X",
    "flatten",
    "make_normalizer",
    "verdict",
    "visible",
]
__version__ = "1.0.0"
''')


In [ ]:
# 모듈: ncs_core/documents.py
write_module('ncs_core/documents.py', r'''"""문서를 한 번 읽어 Document로 만든다. 검수 정책이나 Colab에 의존하지 않는다."""

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from . import hwp, hwpx

Table = dict[str, Any]


@dataclass(frozen=True)
class Document:
    """읽은 문서. 모든 텍스트는 연속 공백이 하나로 정리된 상태다.

    sections: 섹션별 문단 텍스트(표 셀 안 문단 포함, 문서 순서).
    tables:   표 목록(문서 순서). 'styles'는 hwpx일 때만 채워진다.
    images:   (섹션, 문단 인덱스, 원본 그림 이름). hwpx만.
    """

    path: Path
    format: str
    sections: list[list[str]]
    tables: list[Table] = field(default_factory=list)
    images: list[tuple[int, int, str]] = field(default_factory=list)

    @property
    def has_styles(self) -> bool:
        return self.format == "hwpx"

    def section_tables(self, index: int) -> list[Table]:
        return [t for t in self.tables if t["section"] == index]


def detect_format(path) -> str:
    """파일 내용으로 hwp/hwpx를 판별한다. 둘 다 아니면 ValueError."""
    path = Path(path)
    with path.open("rb") as stream:
        head = stream.read(8)
    if head == hwp.OLE_MAGIC:
        if not hwp.is_hwp(path.read_bytes()):
            raise ValueError(f"OLE 파일이지만 HWP 서명이 아님: {path}")
        return "hwp"
    if head[:2] == b"PK":
        if hwpx.is_hwpx(path):
            return "hwpx"
        raise ValueError(f"zip이지만 hwpx가 아님: {path}")
    raise ValueError(f"hwp/hwpx 둘 다 아님 (머리 바이트 {head!r}): {path}")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def load_document(path) -> Document:
    path = Path(path)
    fmt = detect_format(path)
    if fmt == "hwp":
        sections, tables = hwp.load_hwp(path.read_bytes())
        images = []
    else:
        sections, tables, images = hwpx.load_hwpx(path)
    sections = [[_clean(p) for p in section] for section in sections]
    for table in tables:
        table["cells"] = {key: _clean(value) for key, value in table["cells"].items()}
    if not any(line for section in sections for line in section):
        raise ValueError(f"문단 텍스트를 추출하지 못했습니다: {path}")
    return Document(path, fmt, sections, tables, images)
''')


In [ ]:
# 모듈: ncs_core/hwp.py
write_module('ncs_core/hwp.py', r'''"""HWP(바이너리, OLE/CFB) 읽기. 문단 텍스트와 표 셀 텍스트만 읽고 서식은 읽지 않는다."""

import struct
import zlib
from collections.abc import Iterator

OLE_MAGIC = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
HWP_SIGNATURE = b"HWP Document File"
END_OF_CHAIN = (0xFFFFFFFE, 0xFFFFFFFF)

TAG_PARA_TEXT, TAG_LIST_HEADER, TAG_TABLE = 0x43, 0x48, 0x4D
# 문단 텍스트의 제어 문자 중 2바이트짜리. 나머지 제어 문자는 16바이트.
SHORT_CONTROL_CHARS = (0, 10, 13, 24, 25, 26, 27, 28, 29, 30, 31)


def read_cfb(data: bytes):
    """CFB 컨테이너를 열어 (read_stream, 스트림 이름 목록)을 돌려준다.

    섹터 n은 파일 오프셋 (n+1)*섹터크기에서 시작한다(CFB 규격).
    순환하는 섹터 체인은 손상으로 보고 예외를 낸다.
    """
    header = data[:512]
    sector_size = 1 << struct.unpack_from("<H", header, 0x1E)[0]
    mini_sector_size = 1 << struct.unpack_from("<H", header, 0x20)[0]
    dir_start = struct.unpack_from("<I", header, 0x30)[0]
    mini_cutoff = struct.unpack_from("<I", header, 0x38)[0]
    minifat_start = struct.unpack_from("<I", header, 0x3C)[0]
    difat_start = struct.unpack_from("<I", header, 0x44)[0]
    difat = list(struct.unpack_from("<109I", header, 0x4C))
    entries_per_sector = sector_size // 4

    def sector(n: int) -> bytes:
        return data[(n + 1) * sector_size : (n + 2) * sector_size]

    next_difat, seen = difat_start, set()
    while next_difat not in END_OF_CHAIN:
        if next_difat in seen:
            raise ValueError("손상된 HWP: DIFAT 순환 참조")
        seen.add(next_difat)
        values = struct.unpack(f"<{entries_per_sector}I", sector(next_difat))
        difat += values[:-1]
        next_difat = values[-1]

    fat: list[int] = []
    for fat_sector in difat:
        if fat_sector not in END_OF_CHAIN:
            fat += struct.unpack(f"<{entries_per_sector}I", sector(fat_sector))

    def chain(start: int, table: list[int]) -> list[int]:
        out, current, visited = [], start, set()
        while current not in END_OF_CHAIN:
            if current in visited or current >= len(table):
                raise ValueError("손상된 HWP: 잘못된 섹터 체인")
            visited.add(current)
            out.append(current)
            current = table[current]
        return out

    def read_chain(start: int, size: int) -> bytes:
        return b"".join(sector(s) for s in chain(start, fat))[:size]

    directory = b"".join(sector(s) for s in chain(dir_start, fat))
    entries = []
    for i in range(len(directory) // 128):
        entry = directory[i * 128 : (i + 1) * 128]
        name_length = struct.unpack_from("<H", entry, 0x40)[0]
        entries.append(
            (
                entry[: max(name_length - 2, 0)].decode("utf-16le", "ignore"),
                entry[0x42],
                struct.unpack_from("<I", entry, 0x74)[0],
                struct.unpack_from("<Q", entry, 0x78)[0],
            )
        )
    ministream = read_chain(entries[0][2], entries[0][3])
    minifat: list[int] = []
    for s in chain(minifat_start, fat):
        minifat += struct.unpack(f"<{entries_per_sector}I", sector(s))

    def read_stream(name: str) -> bytes | None:
        for entry_name, kind, start, size in entries:
            if entry_name == name and kind == 2:
                if size < mini_cutoff:
                    return b"".join(
                        ministream[s * mini_sector_size : (s + 1) * mini_sector_size]
                        for s in chain(start, minifat)
                    )[:size]
                return read_chain(start, size)
        return None

    return read_stream, [name for name, kind, *_ in entries if kind == 2]


def records(buffer: bytes) -> Iterator[tuple[int, int, bytes]]:
    """본문 스트림을 (tag, level, payload)로 나눈다. 잘린 레코드는 예외."""
    i = 0
    while i + 4 <= len(buffer):
        head = struct.unpack_from("<I", buffer, i)[0]
        i += 4
        tag, level, size = head & 0x3FF, (head >> 10) & 0x3FF, head >> 20
        if size == 0xFFF:
            if i + 4 > len(buffer):
                raise ValueError("손상된 HWP: 확장 레코드 길이 누락")
            size = struct.unpack_from("<I", buffer, i)[0]
            i += 4
        if i + size > len(buffer):
            raise ValueError("손상된 HWP: 레코드 잘림")
        yield tag, level, buffer[i : i + size]
        i += size


def para_text(payload: bytes) -> str:
    """PARA_TEXT 레코드에서 문자열을 꺼낸다. 탭·줄바꿈은 공백으로."""
    out, i, n = [], 0, len(payload)
    while i + 2 <= n:
        code = struct.unpack_from("<H", payload, i)[0]
        if code < 32:
            i += 2 if code in SHORT_CONTROL_CHARS else 16
            if code in (9, 10, 13):
                out.append(" ")
            continue
        out.append(chr(code))
        i += 2
    return "".join(out)


def is_hwp(data: bytes) -> bool:
    """OLE 파일이고 FileHeader 스트림의 서명이 HWP인가."""
    if data[:8] != OLE_MAGIC:
        return False
    try:
        read_stream, _ = read_cfb(data)
        header = read_stream("FileHeader")
    except (ValueError, struct.error, IndexError):
        return False
    return header is not None and header[:32].rstrip(b"\x00") == HWP_SIGNATURE


def load_hwp(data: bytes) -> tuple[list[list[str]], list[dict]]:
    """(섹션별 문단 목록, 표 목록)을 돌려준다.

    문단 목록은 모든 PARA_TEXT 레코드(표 셀 안 포함)를 순서대로 담는다.
    표의 `paragraph`는 그 표를 가진 문단의 인덱스(같은 기준)다.
    """
    read_stream, names = read_cfb(data)
    file_header = read_stream("FileHeader")
    if file_header is None:
        raise ValueError("HWP FileHeader 스트림이 없음")
    flags = struct.unpack_from("<I", file_header, 36)[0]
    if flags & 2:
        raise ValueError("암호화된 hwp는 읽을 수 없음")
    section_names = sorted(
        (n for n in names if n.startswith("Section")), key=lambda s: int(s[7:])
    )
    sections, tables = [], []
    for section_index, name in enumerate(section_names):
        raw = read_stream(name)
        if flags & 1:
            raw = zlib.decompress(raw, -15)
        paragraphs: list[str] = []
        stack: list[tuple[int, dict]] = []  # (level, table)
        cell: tuple[int, dict, tuple[int, int]] | None = None
        for tag, level, payload in records(raw):
            if tag == TAG_PARA_TEXT:
                paragraphs.append(para_text(payload))
                if cell and level == cell[0] + 1:
                    table, key = cell[1], cell[2]
                    table["cells"][key] = (table["cells"][key] + " " + paragraphs[-1]).strip()
            elif tag == TAG_TABLE:
                rows, cols = struct.unpack_from("<HH", payload, 4)
                table = {
                    "section": section_index,
                    "paragraph": max(len(paragraphs) - 1, 0),
                    "rows": rows,
                    "cols": cols,
                    "cells": {},
                    "styles": None,
                }
                while stack and stack[-1][0] >= level:
                    stack.pop()
                stack.append((level, table))
                tables.append(table)
            elif tag == TAG_LIST_HEADER:
                while stack and stack[-1][0] > level:
                    stack.pop()  # 안쪽 표가 끝남
                if stack and stack[-1][0] == level and len(payload) >= 16:
                    col, row = struct.unpack_from("<HH", payload, 8)
                    table = stack[-1][1]
                    table["cells"].setdefault((row, col), "")
                    cell = (level, table, (row, col))
                else:
                    cell = None
        sections.append(paragraphs)
    return sections, tables
''')


In [ ]:
# 모듈: ncs_core/hwpx.py
write_module('ncs_core/hwpx.py', r'''"""HWPX(zip+XML) 읽기. 표준 ElementTree만 사용한다.

한 섹션 XML을 한 번 순회해 문단 텍스트, 표(셀 텍스트와 서식), 그림 이름을 함께 꺼낸다.
"""

import re
import zipfile
from xml.etree import ElementTree as ET

HP = "{http://www.hancom.co.kr/hwpml/2011/paragraph}"
HH = "{http://www.hancom.co.kr/hwpml/2011/head}"
HC = "{http://www.hancom.co.kr/hwpml/2011/core}"

SECTION_NAME = re.compile(r"Contents/section(\d+)\.xml")
IMAGE_NAME = re.compile(r"원본 그림의 이름:\s*([^\n]+)")
NO_FILL = (None, "", "none", "#FFFFFF", "#ffffff")


def is_hwpx(path) -> bool:
    with zipfile.ZipFile(path) as archive:
        return any(SECTION_NAME.fullmatch(n) for n in archive.namelist())


def section_names(archive: zipfile.ZipFile) -> list[str]:
    names = [n for n in archive.namelist() if SECTION_NAME.fullmatch(n)]
    return sorted(names, key=lambda n: int(SECTION_NAME.fullmatch(n).group(1)))


def para_text(paragraph: ET.Element) -> str:
    """변경추적 반영본 텍스트. deleteBegin~deleteEnd 구간은 버린다.

    표 셀·캡션 안에 중첩된 <hp:p>는 내려가지 않는다. 그 문단들은 섹션 순회에서
    따로 나오므로, 내려가면 바깥 문단에 셀 텍스트가 전부 합쳐진다.
    <hp:t> 안 인라인 요소 뒤의 tail 텍스트는 포함한다.
    """
    out, state = [], None

    def walk(element: ET.Element) -> None:
        nonlocal state
        for child in element:
            tag = child.tag.replace(HP, "")
            if tag == "p":
                continue
            if tag == "deleteBegin":
                state = "del"
            elif tag in ("deleteEnd", "insertEnd"):
                state = None
            elif tag == "insertBegin":
                state = "ins"
            if state != "del" and tag == "t":
                out.append(child.text or "")
            walk(child)
            if state != "del" and child.tail and element.tag == HP + "t":
                out.append(child.tail)

    walk(paragraph)
    return "".join(out)


def header_styles(archive: zipfile.ZipFile) -> tuple[dict, dict]:
    """header.xml → (charPr id→{bold, font}, borderFill id→면색 또는 None)."""
    root = ET.fromstring(archive.read("Contents/header.xml"))
    fonts = {}
    for face in root.iter(HH + "fontface"):
        if face.get("lang") == "HANGUL":
            for font in face.findall(HH + "font"):
                fonts[font.get("id")] = font.get("face")
    char_props = {}
    for prop in root.iter(HH + "charPr"):
        ref = prop.find(HH + "fontRef")
        char_props[prop.get("id")] = {
            "bold": prop.find(HH + "bold") is not None,
            "font": fonts.get(ref.get("hangul")) if ref is not None else None,
        }
    fills = {}
    for border_fill in root.iter(HH + "borderFill"):
        brush = border_fill.find(".//" + HC + "winBrush")
        color = brush.get("faceColor") if brush is not None else None
        fills[border_fill.get("id")] = None if color in NO_FILL else color.upper()
    return char_props, fills


def _cell_runs(cell: ET.Element, own_paragraphs: list[ET.Element], char_props: dict) -> list[dict]:
    runs = []
    for paragraph in own_paragraphs:
        for run in paragraph.iter(HP + "run"):
            text = re.sub(r"\s+", " ", "".join(t.text or "" for t in run.iter(HP + "t"))).strip()
            if text:
                runs.append(
                    {
                        "text": text,
                        **char_props.get(run.get("charPrIDRef"), {"bold": False, "font": None}),
                    }
                )
    return runs


def parse_section(root: ET.Element, section_index: int, styles: tuple[dict, dict] | None):
    """(문단 텍스트 목록, 표 목록, 그림 목록)을 돌려준다.

    표: {'section', 'paragraph', 'rows', 'cols', 'cells': {(r,c): 텍스트},
         'styles': {(r,c): {'fill', 'runs': [{'text','font','bold'}]}} (styles 없으면 None)}
    그림: (문단 인덱스, 원본 그림 이름)
    """
    parents = {child: parent for parent in root.iter() for child in parent}
    paragraphs = list(root.iter(HP + "p"))
    positions = {p: i for i, p in enumerate(paragraphs)}

    def owner_paragraph(element: ET.Element) -> ET.Element | None:
        current = parents.get(element)
        while current is not None and current.tag != HP + "p":
            current = parents.get(current)
        return current

    def owner_cell(paragraph: ET.Element) -> ET.Element | None:
        current = parents.get(paragraph)
        while current is not None and current.tag != HP + "tc":
            current = parents.get(current)
        return current

    images = []
    for comment in root.iter(HP + "shapeComment"):
        match = IMAGE_NAME.search(comment.text or "")
        paragraph = owner_paragraph(comment)
        if match and paragraph in positions:
            images.append((positions[paragraph], match.group(1).strip()))

    tables = []
    for table in root.iter(HP + "tbl"):
        paragraph = owner_paragraph(table)
        if paragraph not in positions:
            continue
        cells, cell_styles = {}, {}
        for row in table.findall(HP + "tr"):
            for cell in row.findall(HP + "tc"):
                address = cell.find(HP + "cellAddr")
                if address is None:
                    raise ValueError("HWPX 표 셀에 cellAddr가 없습니다")
                key = (int(address.get("rowAddr")), int(address.get("colAddr")))
                # 중첩 표는 자체 표로 처리하므로 바깥 셀에 합치지 않는다.
                own = [p for p in cell.iter(HP + "p") if owner_cell(p) is cell]
                cells[key] = re.sub(r"\s+", " ", " ".join(para_text(p) for p in own)).strip()
                if styles is not None:
                    char_props, fills = styles
                    cell_styles[key] = {
                        "fill": fills.get(cell.get("borderFillIDRef")),
                        "runs": _cell_runs(cell, own, char_props),
                    }
        tables.append(
            {
                "section": section_index,
                "paragraph": positions[paragraph],
                "rows": int(table.get("rowCnt") or 0),
                "cols": int(table.get("colCnt") or 0),
                "cells": cells,
                "styles": cell_styles if styles is not None else None,
            }
        )
    return [para_text(p) for p in paragraphs], tables, images


def load_hwpx(path):
    """(섹션별 문단, 표, 그림) — 그림은 (섹션, 문단, 이름)."""
    sections, tables, images = [], [], []
    with zipfile.ZipFile(path) as archive:
        try:
            styles = header_styles(archive)
        except KeyError:
            styles = None
        for index, name in enumerate(section_names(archive)):
            root = ET.fromstring(archive.read(name))
            paragraphs, section_tables, section_images = parse_section(root, index, styles)
            sections.append(paragraphs)
            tables.extend(section_tables)
            images.extend((index, paragraph, image) for paragraph, image in section_images)
    return sections, tables, images
''')


In [ ]:
# 모듈: ncs_core/inputs.py
write_module('ncs_core/inputs.py', r'''"""입력 파일의 역할 판별과 검증. Colab UI와 독립적이다."""

import hashlib
from pathlib import Path

from .documents import Document, load_document

ROLE_LABELS = {"manuscript": "원고", "plan": "개발계획서"}
MANUSCRIPT_MARKERS = ("차 례", "차례", "학습모듈의 개요", "NCS-학습모듈의 위치")


def detect_role(sections: list[list[str]], head_lines: int = 60) -> str:
    """첫 섹션 머리 문단으로 역할을 판별한다: 'plan' / 'manuscript' / 'unknown'."""
    head = " ".join(sections[0][:head_lines]) if sections else ""
    if "개발계획서" in head:
        return "plan"
    if any(marker in head for marker in MANUSCRIPT_MARKERS):
        return "manuscript"
    return "unknown"


def file_digest(path) -> bytes:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.digest()


def validate_distinct(paths: dict[str, str | Path]) -> None:
    """역할별 경로가 존재하고 비어 있지 않으며 서로 다른 내용인지 확인한다."""
    for role, path in paths.items():
        label = ROLE_LABELS.get(role, role)
        if not path:
            raise ValueError(f"{label} 파일 경로를 지정하세요")
        path = Path(path)
        if not path.is_file():
            raise ValueError(f"{label} 파일을 찾을 수 없습니다: {path}")
        if path.stat().st_size == 0:
            raise ValueError(f"{label} 파일이 비어 있습니다: {path}")
    digests = {role: file_digest(path) for role, path in paths.items()}
    if len(set(digests.values())) != len(digests):
        raise ValueError("입력 파일들의 내용이 동일합니다. 실제 원고와 개발계획서를 각각 선택하세요")


def select_documents(
    paths,
    roles: dict | None = None,
    *,
    required: tuple[str, ...] = ("manuscript",),
) -> dict[str, Document]:
    """경로 목록을 읽어 {'manuscript': Document, 'plan': Document}로 나눈다.

    roles={경로: 'plan'|'manuscript'}로 자동 판별을 덮어쓴다.
    required에 있는 역할이 없거나, 한 역할에 파일이 둘 이상이면 ValueError.
    """
    overrides = {str(Path(p).resolve()): r for p, r in (roles or {}).items()}
    if set(overrides.values()) - set(ROLE_LABELS):
        raise ValueError("roles 값은 'manuscript' 또는 'plan'이어야 합니다")
    documents: dict[str, Document] = {}
    seen: set[str] = set()
    for path in paths:
        key = str(Path(path).resolve())
        if key in seen:
            raise ValueError(f"같은 입력 파일이 중복 지정되었습니다: {path}")
        seen.add(key)
        document = load_document(path)
        role = overrides.get(key, detect_role(document.sections))
        if role not in ROLE_LABELS:
            raise ValueError(f"파일 역할을 판단하지 못했습니다: {path}. roles로 역할을 지정하세요")
        if role in documents:
            raise ValueError(f"{ROLE_LABELS[role]} 파일이 둘 이상입니다. 역할마다 한 개씩 선택하세요")
        documents[role] = document
    unused = overrides.keys() - seen
    if unused:
        raise ValueError(f"roles에 입력 목록에 없는 경로가 있습니다: {sorted(unused)}")
    for role in required:
        if role not in documents:
            raise ValueError(f"{ROLE_LABELS[role]} 파일이 필요합니다")
    if len(documents) > 1:
        validate_distinct({role: doc.path for role, doc in documents.items()})
    return documents
''')


In [ ]:
# 모듈: ncs_core/models.py
write_module('ncs_core/models.py', r'''"""파이프라인 경계의 공통 데이터 계약. 체크리스트 행의 한국어 JSON 키는 기존과 같다."""

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal, TypedDict

ChecklistItem = TypedDict(
    "ChecklistItem",
    {
        "item_id": str,
        "region_path": str,
        "원고": Any,
        "체크": dict[str, Literal["O", "X", "△", "-"]],
        "판정": str,
        "비고": dict[str, str] | list[str],
        "수정 제안": Any,
    },
)


@dataclass
class ReviewResult:
    """한 번의 검사 결과. 저장 형식과 무관하다."""

    module_code: str
    checklist: list[ChecklistItem]
    findings: list[dict]
    diagnostics: dict[str, Any] = field(default_factory=dict)
    context: Any = None


@dataclass(frozen=True)
class SavedReport:
    directory: Path
    zip_path: Path
    summary: dict[str, Any]
''')


In [ ]:
# 모듈: ncs_core/module_code.py
write_module('ncs_core/module_code.py', r'''"""모듈 코드 검증과 파일명 추출. input()을 호출하지 않는다."""

import re
from pathlib import Path

CODE_PATTERN = r"LM[0-9]{10}_[0-9]+v[0-9]+"
CODE_IN_NAME = re.compile(r"(?<![A-Za-z0-9])" + CODE_PATTERN + r"(?![A-Za-z0-9])")


def validate_module_code(code: str) -> str:
    code = (code or "").strip()
    if not re.fullmatch(CODE_PATTERN, code):
        raise ValueError("모듈 코드는 LM + 숫자 10자리 + _ + 숫자 + v + 숫자 형식이어야 합니다")
    if int(code[8:10]) < 1 or int(code[10:12]) < 1:
        raise ValueError("세분류 번호와 능력단위 번호는 01 이상이어야 합니다")
    return code


def resolve_module_code(manuscript_path, override: str | None = "") -> str:
    """override가 있으면 검증해 쓰고, 없으면 원고 파일명에서 하나뿐인 코드를 찾는다."""
    if override and override.strip():
        return validate_module_code(override)
    candidates = list(dict.fromkeys(CODE_IN_NAME.findall(Path(manuscript_path).name)))
    if len(candidates) != 1:
        raise ValueError("원고 파일명의 모듈 코드가 없거나 여러 개입니다. module_code를 지정하세요")
    return validate_module_code(candidates[0])
''')


In [ ]:
# 모듈: ncs_core/reporting.py
write_module('ncs_core/reporting.py', r'''"""검사 결과를 항목별 JSON과 ZIP으로 저장한다. 파일 이름 규칙은 두 패키지가 같다.

<out_dir>/<코드>_<실행별 폴더>/
    <코드>_<항목 꼬리말>.json   항목별 체크리스트
    <코드>_<이름>_00_요약.json  검수결과(적합/부적합/판단불가) 종합과 항목별 집계
    <코드>_<이름>_지적사항.json  X·△만
    <코드>_<이름>_전체.json     모든 체크리스트 행
    <코드>_<이름>_진단.json     입력·읽기 상태
    <코드>_<이름>_검수결과.zip
"""

import json
import tempfile
import zipfile
from collections.abc import Sequence
from pathlib import Path

from .models import ReviewResult, SavedReport
from .module_code import validate_module_code
from .text import FAIL, PASS, UNDECIDED, VERDICTS


def overall_verdict(verdicts) -> str:
    """행 판정들의 종합: 부적합이 하나라도 있으면 부적합, 없고 판단불가가 있으면 판단불가, 나머지는 적합."""
    verdicts = list(verdicts)
    if FAIL in verdicts:
        return FAIL
    if UNDECIDED in verdicts:
        return UNDECIDED
    return PASS


def save_report(
    result: ReviewResult,
    out_dir,
    *,
    name: str,
    item_files: Sequence[tuple[str, str]],
    other_tail: str,
    per_run_dir: bool = True,
) -> SavedReport:
    """name: 결과 이름('차례'/'위치'). item_files: (item_id 접두, 파일 꼬리말).

    per_run_dir=True이면 실행마다 새 하위 폴더를 만들어 이전 결과를 덮어쓰지 않는다.
    False이면 out_dir에 바로 쓰고 같은 코드의 이전 결과를 갱신한다.
    item_files에 없는 item_id는 other_tail 파일로 모인다.
    """
    code = validate_module_code(result.module_code)
    root = Path(out_dir).resolve()
    root.mkdir(parents=True, exist_ok=True)
    directory = Path(tempfile.mkdtemp(prefix=f"{code}_", dir=root)) if per_run_dir else root

    buckets: dict[str, list] = {tail: [] for _, tail in item_files}
    by_length = sorted(item_files, key=lambda pair: -len(pair[0]))
    for row in result.checklist:
        tail = next((t for prefix, t in by_length if row["item_id"].startswith(prefix)), other_tail)
        buckets.setdefault(tail, []).append(row)

    paths: list[Path] = []
    summary: dict = {}

    def write(tail: str, value) -> Path:
        path = directory / f"{code}_{tail}.json"
        path.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")
        paths.append(path)
        return path

    for tail, rows in buckets.items():
        path = write(tail, rows)
        verdicts = [row["판정"] for row in rows]
        summary[tail] = {"검수결과": overall_verdict(verdicts), "행 수": {v: verdicts.count(v) for v in VERDICTS}, "파일": path.name}
    summary = {"검수결과": overall_verdict(row["판정"] for row in result.checklist), "항목": summary}
    write(f"{name}_00_요약", summary)
    write(f"{name}_지적사항", result.findings)
    write(f"{name}_전체", result.checklist)
    write(f"{name}_진단", result.diagnostics)

    zip_path = directory / f"{code}_{name}_검수결과.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in paths:
            archive.write(path, path.name)
    return SavedReport(directory, zip_path, summary)
''')


In [ ]:
# 모듈: ncs_core/rules.py
write_module('ncs_core/rules.py', r'''"""규칙 등록과 실행. 규칙 함수는 컨텍스트 하나를 받아 체크리스트 행 목록을 돌려준다."""

from collections.abc import Callable, Sequence
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Rule:
    """key: 규칙 식별자(중복 불가). item_prefix: 반환 행의 item_id가 시작해야 하는 문자열."""

    key: str
    item_prefix: str
    run: Callable[[Any], list[dict]]


def run_rules(context: Any, rules: Sequence[Rule]) -> list[dict]:
    keys = [rule.key for rule in rules]
    if len(keys) != len(set(keys)):
        raise ValueError("중복된 규칙 key가 있습니다")
    rows: list[dict] = []
    for rule in rules:
        result = rule.run(context)
        for row in result:
            if not row["item_id"].startswith(rule.item_prefix):
                raise ValueError(f"{rule.key} 규칙이 다른 item_id를 반환했습니다: {row['item_id']}")
        rows.extend(result)
    return rows
''')


In [ ]:
# 모듈: ncs_core/tables.py
write_module('ncs_core/tables.py', r'''"""Document.tables(문서 순서의 표 목록)를 다루는 작은 도우미."""

from .documents import Table


def column(table: Table, col: int) -> list[str]:
    """해당 열의 비어 있지 않은 셀 텍스트를 행 순서대로."""
    return [v for (r, c), v in sorted(table["cells"].items()) if c == col and v]


def column_cells(table: Table, col: int, header: str | None = None) -> list[tuple[int, str]]:
    """해당 열의 비어 있지 않은 셀을 (행, 텍스트)로, header 텍스트는 제외."""
    return [
        (r, v)
        for (r, c), v in sorted(table["cells"].items())
        if c == col and v and v != header
    ]


def header_columns(table: Table, names: tuple[str, ...]) -> dict[str, int]:
    """names에 있는 머리글 셀 텍스트 → 열 번호 (먼저 나온 것 우선)."""
    found: dict[str, int] = {}
    for (r, c), value in sorted(table["cells"].items()):
        if value in names and value not in found:
            found[value] = c
    return found


def find_table(tables: list[Table], must_contain: str, after: Table | None = None) -> Table | None:
    """must_contain을 포함한 셀이 있는 첫 표 (after 표 다음부터)."""
    start = tables.index(after) + 1 if after is not None else 0
    for table in tables[start:]:
        if any(must_contain in v for v in table["cells"].values()):
            return table
    return None
''')


In [ ]:
# 모듈: ncs_core/text.py
write_module('ncs_core/text.py', r'''"""문자 정규화·비교·판정·지적사항 변환. 기존 결과와 호환된다.

정규화에서 어떤 머리 번호를 지우는지는 패키지마다 다르므로
`make_normalizer(prefix)`로 각 패키지가 자기 규칙의 `norm`/`same`을 만든다.
"""

import re
from collections.abc import Callable, Iterable

PUA = re.compile(r"[-]")
O, X, TRI, NA = "O", "X", "△", "-"
# 검수 확인서의 검수결과 표현. 한 항목의 판정은 이 셋 중 하나다.
PASS, FAIL, UNDECIDED = "적합", "부적합", "판단불가"
VERDICTS = (PASS, FAIL, UNDECIDED)

Normalizer = Callable[[str | None], str]


def make_normalizer(prefix: str | None = None) -> tuple[Normalizer, Callable[..., bool]]:
    """(norm, same)을 만든다.

    norm: 기호 글꼴(PUA) 문자를 가운뎃점으로 바꾸고, 양끝 공백과 머리 번호(prefix)를
    지운 뒤 연속 공백을 하나로 만든다.
    same: 두 문자열의 norm 결과가 같은가.
    """
    head = re.compile(prefix) if prefix else None

    def norm(text: str | None) -> str:
        value = PUA.sub("·", text or "").strip()
        if head is not None:
            value = head.sub("", value)
        return re.sub(r"\s+", " ", value)

    def same(expected: str | None, actual: str | None) -> bool:
        return norm(expected) == norm(actual)

    return norm, same


def visible(text: str | None) -> str:
    """PUA 문자를 ⟨U+XXXX⟩로 드러내 결과에 적는다."""
    return PUA.sub(lambda m: f"⟨U+{ord(m.group()):04X}⟩", text or "")


def verdict(checks: dict[str, str]) -> str:
    """X가 있으면 부적합, X 없이 △가 있으면 판단불가, 나머지는 적합. '-'는 무시."""
    values = [v for v in checks.values() if v != NA]
    if X in values:
        return FAIL
    if TRI in values:
        return UNDECIDED
    return PASS


def flatten(checklist: Iterable[dict]) -> list[dict]:
    """체크 항목 중 X·△만 지적사항 행으로 편다 (기존 키 유지)."""
    findings = []
    for row in checklist:
        for name, value in row["체크"].items():
            if value not in (X, TRI):
                continue
            notes = row["비고"]
            reason = notes.get(name, "") if isinstance(notes, dict) else "; ".join(notes)
            findings.append(
                {
                    "item_id": row["item_id"],
                    "check": name,
                    "region_path": row["region_path"],
                    "verdict": FAIL if value == X else UNDECIDED,
                    "evidence_quote": row["원고"],
                    "reason": reason,
                    "suggestion": row["수정 제안"],
                }
            )
    return findings
''')


In [ ]:
# 모듈: ncs_overview/__init__.py
write_module('ncs_overview/__init__.py', r'''"""F02 학습모듈의 개요 ①·②·⑤·⑥·⑧ 검수. 읽기·판정·저장은 ncs_core를 쓴다."""

from ncs_core import ReviewResult, SavedReport

from .config import CheckConfig
from .pipeline import evaluate_review, prepare_review, run_review
from .registry import DEFAULT_RULES
from .reporting import save_results

__all__ = [
    "CheckConfig",
    "DEFAULT_RULES",
    "ReviewResult",
    "SavedReport",
    "prepare_review",
    "evaluate_review",
    "run_review",
    "save_results",
]
__version__ = "1.0.0"
''')


In [ ]:
# 모듈: ncs_overview/__main__.py
write_module('ncs_overview/__main__.py', r'''"""python -m ncs_overview 원고.hwpx [개발계획서.hwp] [--module-code LM...] [--out 결과]"""

import argparse
import json

from . import CheckConfig, run_review, save_results


def main() -> None:
    parser = argparse.ArgumentParser(description="F02 학습모듈의 개요 ①·②·⑤·⑥·⑧ 검수")
    parser.add_argument("paths", nargs="+", help="원고와 (선택) 개발계획서. 순서 무관, 내용으로 역할 판별")
    parser.add_argument("--manuscript", help="자동 역할 판별 대신 원고로 지정할 입력 경로")
    parser.add_argument("--plan", help="자동 역할 판별 대신 개발계획서로 지정할 입력 경로")
    parser.add_argument("--module-code", default="", help="비우면 원고 파일명에서 추출")
    parser.add_argument("--front-section", type=int, default=0)
    parser.add_argument("--out", default="결과")
    args = parser.parse_args()
    roles = {}
    if args.manuscript:
        roles[args.manuscript] = "manuscript"
    if args.plan:
        roles[args.plan] = "plan"
    try:
        config = CheckConfig(front_section=args.front_section)
        result = run_review(args.paths, roles=roles, module_code=args.module_code, config=config)
        saved = save_results(result, args.out)
    except (OSError, ValueError) as error:
        parser.exit(2, f"검수 실행 실패: {error}\n")
    print(json.dumps({"요약": saved.summary, "진단": result.diagnostics}, ensure_ascii=False, indent=2))
    print(f"지적사항 {len(result.findings)}건 · {saved.zip_path}")


if __name__ == "__main__":
    main()
''')


In [ ]:
# 모듈: ncs_overview/compare.py
write_module('ncs_overview/compare.py', r'''"""목록 비교 도우미. ncs_toc.compare.diff_list와 같은 정책(difflib, 누락/추가/변경/순서 구분)을 개요 정규화 규칙으로 쓴다."""

import difflib
from collections.abc import Sequence

from ncs_core import O, X, visible

from .text import norm


def diff_list(label: str, expected: Sequence[str], actual: Sequence[str]) -> tuple[str, list[str]]:
    exp, act = [norm(x) for x in expected], [norm(x) for x in actual]
    notes: list[str] = []
    if len(expected) != len(actual):
        notes.append(f"{label} 개수: 원고 {len(actual)} / 개발계획서 {len(expected)}")
    if exp != act and sorted(exp) == sorted(act):
        notes.append(f"{label} 순서 다름: 원고 {[visible(x) for x in actual]} / 개발계획서 {list(expected)}")
        return X, notes
    matcher = difflib.SequenceMatcher(None, exp, act, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            continue
        if op == "replace" and i2 - i1 == j2 - j1:
            for k in range(i2 - i1):
                notes.append(f'{label} {i1 + k + 1}번째 다름: "{visible(actual[j1 + k])}" / 개발계획서 "{expected[i1 + k]}"')
        else:
            for k in range(i1, i2):
                notes.append(f'{label} {k + 1}번째 누락: "{expected[k]}"')
            for k in range(j1, j2):
                notes.append(f'{label} 원고 {k + 1}번째가 개발계획서에 없음: "{visible(actual[k])}"')
    return (O if not notes else X), notes
''')


In [ ]:
# 모듈: ncs_overview/config.py
write_module('ncs_overview/config.py', r'''"""개요 검수의 조정 가능한 양식 가정."""
from dataclasses import dataclass


@dataclass(frozen=True)
class CheckConfig:
    front_section: int = 0          # 학습모듈의 개요가 있는 섹션
    max_prerequisites: int = 5      # ② 선수학습 최대 개수
    max_terms: int = 10             # ⑧ 핵심 용어 최대 개수
    term_max_length: int = 25       # ⑧ 이보다 긴 항목은 단어가 아닐 가능성 → △

    def __post_init__(self):
        if self.front_section < 0:
            raise ValueError("섹션 번호는 0 이상이어야 합니다")
        for name in ("max_prerequisites", "max_terms", "term_max_length"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name}은 1 이상이어야 합니다")


DEFAULT_CONFIG = CheckConfig()
''')


In [ ]:
# 모듈: ncs_overview/models.py
write_module('ncs_overview/models.py', r'''"""파서와 규칙 사이의 경계. 결과 모델(ReviewResult 등)은 ncs_core.models를 쓴다."""

from dataclasses import dataclass
from typing import Any

from ncs_core import Document

from .config import CheckConfig


@dataclass(frozen=True)
class ReviewContext:
    manuscript: Document
    plan: Document | None
    module_code: str
    config: CheckConfig
    overview: dict[str, Any]          # parsers.parse_overview 결과
    plan_info: dict[str, Any] | None  # parsers.parse_plan 결과 (계획서 없으면 None)
''')


In [ ]:
# 모듈: ncs_overview/parsers.py
write_module('ncs_overview/parsers.py', r'''"""학습모듈의 개요(목표·선수학습·내용체계·핵심 용어)와 개발계획서 모듈 블록을 추출한다.

내용체계는 표 셀(hwp·hwpx 모두 표 셀 텍스트가 있다)을 우선 읽고, 표를 못 찾으면 문단 줄로 읽는다.
차례 검수(ncs_toc.parsers)의 OV_LESSON/OV_SUB 규칙과 같은 정규식을 쓴다.
"""

import re

from ncs_core import Document, Table

from .patterns import (
    ELEMENT_CODE,
    HEAD_GOAL,
    HEAD_OVERVIEW,
    HEAD_PREREQ,
    HEAD_STRUCTURE,
    HEAD_TERMS,
    OV_LESSON,
    OV_SUB,
    PLAN_LESSON,
)
from .text import split_items

HEADINGS = (HEAD_GOAL, HEAD_PREREQ, HEAD_STRUCTURE, HEAD_TERMS)
STRUCTURE_HEADERS = ("학습", "학습 내용", "코드번호", "요소 명칭")


def _block(lines: list[tuple[int, str]], head, stop_heads) -> tuple[int | None, list[tuple[int, str]]]:
    """head 제목 다음부터 다음 제목(stop_heads) 전까지의 (문단 인덱스, 텍스트)."""
    start = next((k for k, (_, l) in enumerate(lines) if head.match(l)), None)
    if start is None:
        return None, []
    out = []
    for i, l in lines[start + 1 :]:
        if any(h.match(l) for h in stop_heads):
            break
        out.append((i, l))
    return lines[start][0], out


def _structure_from_table(table: Table) -> dict | None:
    cells = table["cells"]
    headers: dict[str, tuple[int, int]] = {}
    for (r, c), v in sorted(cells.items()):
        key = v.replace(" ", "")
        for name in STRUCTURE_HEADERS:
            if key == name.replace(" ", "") and name not in headers:
                headers[name] = (r, c)
    if not {"학습", "코드번호"} <= headers.keys():
        return None
    header_row = max(r for r, _ in headers.values())
    col = {name: c for name, (_, c) in headers.items()}
    content_col = col.get("학습 내용", col["학습"] + 1)
    name_col = col.get("요소 명칭", col["코드번호"] + 1)
    lessons, contents, elements = [], [], []
    rows = sorted({r for r, _ in cells if r > header_row})
    for r in rows:
        lesson, content = cells.get((r, col["학습"]), ""), cells.get((r, content_col), "")
        code, name = cells.get((r, col["코드번호"]), ""), cells.get((r, name_col), "")
        if m := OV_LESSON.match(lesson):
            lessons.append({"no": int(m.group(1)), "text": m.group(2).strip(), "row": r})
        if m := OV_SUB.match(content):
            contents.append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip(), "row": r})
        for part in code.split():
            if ELEMENT_CODE.match(part) and not any(e["code"] == part for e in elements):
                elements.append({"code": part, "name": name, "row": r})
    return {"학습": lessons, "내용": contents, "요소": elements, "근거": f"표 (p{table['paragraph']})"}


def _structure_from_lines(block: list[tuple[int, str]]) -> dict:
    lessons, contents, elements = [], [], []
    for k, (i, l) in enumerate(block):
        if (m := OV_LESSON.match(l)) and int(m.group(1)) == len(lessons) + 1:
            lessons.append({"no": int(m.group(1)), "text": m.group(2).strip(), "row": i})
        elif m := OV_SUB.match(l):
            text = m.group(3).strip() or (block[k + 1][1] if k + 1 < len(block) else "")
            contents.append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": text, "row": i})
        elif m := ELEMENT_CODE.match(l):
            # 요소 명칭은 다음 줄(셀 안에서 줄이 나뉘면 두 줄)
            name_parts = []
            for _, nxt in block[k + 1 : k + 3]:
                if OV_LESSON.match(nxt) or OV_SUB.match(nxt) or ELEMENT_CODE.match(nxt):
                    break
                name_parts.append(nxt)
            elements.append({"code": l, "name": " ".join(name_parts).strip(), "row": i})
    return {"학습": lessons, "내용": contents, "요소": elements, "근거": "문단 줄"}


def parse_overview(document: Document, section: int) -> dict:
    """→ {'목표': {'text','p'}|None, '선수학습': {'items','text','p'}|None,
         '내용체계': {...}|None, '핵심 용어': {'items','text','p'}|None, '개요 제목 p': int|None}"""
    raw = document.sections[section] if section < len(document.sections) else []
    lines = [(i, l) for i, l in enumerate(raw) if l]
    head = next((i for i, l in lines if HEAD_OVERVIEW.search(l) and not re.search(r"\d+$", l)), None)
    if head is not None:
        lines = [(i, l) for i, l in lines if i > head]

    out: dict = {"개요 제목 p": head}
    p, block = _block(lines, HEAD_GOAL, HEADINGS)
    out["목표"] = {"text": " ".join(l for _, l in block), "p": block[0][0]} if block else None
    p, block = _block(lines, HEAD_PREREQ, HEADINGS)
    if block:
        text = " ".join(l for _, l in block)
        out["선수학습"] = {"items": split_items(text), "text": text, "p": block[0][0]}
    else:
        out["선수학습"] = None
    p, block = _block(lines, HEAD_STRUCTURE, HEADINGS)
    structure = None
    if p is not None:
        tables = [t for t in document.section_tables(section) if t["paragraph"] >= p]
        structure = next((s for t in tables if (s := _structure_from_table(t))), None)
        if structure is None and block:
            structure = _structure_from_lines(block)
    out["내용체계"] = structure
    p, block = _block(lines, HEAD_TERMS, HEADINGS)
    if block:
        text = " ".join(l for _, l in block)
        out["핵심 용어"] = {"items": split_items(text), "text": text, "p": block[0][0]}
    else:
        out["핵심 용어"] = None
    return out


def parse_plan(document: Document, module_code: str) -> dict | None:
    """개발계획서 모듈 블록 → {'정의','목표','학습명':[...],'내용명':[{no,sub,text}],'요소':[{code,name}]}.

    블록은 '능력단위 코드번호' 다음 줄의 코드부터 'NCS (수정 금지)' 전까지 (ncs_toc.parsers.parse_plan_toc와 같은 기준).
    """
    lines = [l for section in document.sections for l in section]
    code = module_code.replace("LM", "")
    start = next(
        (i for i, l in enumerate(lines) if l == code and i > 0 and lines[i - 1].startswith("능력단위 코드번호")),
        None,
    )
    if start is None:
        return None
    end = next((i for i in range(start, len(lines)) if lines[i].startswith("NCS (수정 금지)")), len(lines))
    block = [l for l in lines[start:end] if l]
    plan = {"정의": None, "목표": None, "학습명": [], "내용명": [], "요소": []}
    for i, line in enumerate(block):
        nxt = block[i + 1] if i + 1 < len(block) else ""
        if re.fullmatch(r"능력단위\s*정의", line) and plan["정의"] is None:
            plan["정의"] = nxt
        elif re.fullmatch(r"학습모듈의\s*목표", line) and plan["목표"] is None:
            plan["목표"] = nxt
        elif (m := ELEMENT_CODE.match(line)) and m.group(1) == code:
            plan["요소"].append({"code": line, "name": nxt})
        elif m := OV_SUB.match(line):
            text = m.group(3).strip() or nxt
            plan["내용명"].append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": text})
        elif (m := PLAN_LESSON.match(line)) and int(m.group(1)) == len(plan["학습명"]) + 1:
            plan["학습명"].append(m.group(2).strip())
    return plan
''')


In [ ]:
# 모듈: ncs_overview/patterns.py
write_module('ncs_overview/patterns.py', r'''"""개요 양식의 제목·내용 인식 패턴. 판정 정책은 rules/에 둔다."""
import re

HEAD_OVERVIEW = re.compile(r"학습모듈의\s*개요$")
HEAD_GOAL = re.compile(r"^학습모듈의\s*목표$")
HEAD_PREREQ = re.compile(r"^선수\s*학습$")
HEAD_STRUCTURE = re.compile(r"^학습모듈의\s*내용\s*체계$")
HEAD_TERMS = re.compile(r"^핵심\s*용어$")
OV_LESSON = re.compile(r"^(\d+)\.\s*(\S.*)$")           # 1. 명 (개요 표)
PLAN_LESSON = re.compile(r"^(\d+)\.\s+(\S.*)$")         # 계획서 학습명: 공백 필수 ("1.1 …" 수행준거와 구별)
OV_SUB = re.compile(r"^(\d+)-(\d+)\.\s*(.*)$")          # 1-1. 명 (명이 다음 줄에 올 수도 있음)
ELEMENT_CODE = re.compile(r"^(\d{10}_\d+v\d+)\.(\d+)$")  # 1203020201_24v3.1
GOAL_END = re.compile(r"할\s*수\s*있다\s*\.$")
# 능력단위 정의 → 학습모듈의 목표: 주어('…란/이란/은/는') 삭제, '~하는 능력이다' → '~할 수 있다'
DEFINITION_SUBJECT = re.compile(r"^.*?(?:이란|란|은|는)\s+")
DEFINITION_END = re.compile(r"(하|할 수 있)는\s*능력이다\s*\.?$")
# ⑧ 단어(명사형)가 아닐 가능성이 있는 종결
TERM_PREDICATE = re.compile(r"(다|하기|함|됨|하는|되는|할 것|것)$")
''')


In [ ]:
# 모듈: ncs_overview/pipeline.py
write_module('ncs_overview/pipeline.py', r'''"""입력 선택 → 추출 → 규칙 실행. 저장과 화면 출력은 호출자가 결정한다. (ncs_toc.pipeline과 같은 구조)"""

import re
from collections.abc import Sequence
from typing import Any

from ncs_core import Document, ReviewResult, Rule, resolve_module_code, run_rules, select_documents
from ncs_core.text import flatten

from .config import DEFAULT_CONFIG, CheckConfig
from .models import ReviewContext
from .parsers import parse_overview, parse_plan
from .registry import ALL_ITEM_IDS, DEFAULT_RULES


def prepare_review(
    manuscript: Document,
    plan: Document | None = None,
    *,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
) -> ReviewContext:
    code = resolve_module_code(manuscript.path, module_code)
    if config.front_section >= len(manuscript.sections):
        raise ValueError("지정한 개요 섹션이 없습니다. 문서 구조와 CheckConfig를 확인하세요")
    overview = parse_overview(manuscript, config.front_section)
    if overview["개요 제목 p"] is None and not any(overview.get(k) for k in ("목표", "선수학습", "내용체계", "핵심 용어")):
        raise ValueError("학습모듈의 개요를 읽지 못했습니다. 양식과 섹션 설정을 확인하세요")
    plan_info = parse_plan(plan, code) if plan else None
    return ReviewContext(manuscript, plan, code, config, overview, plan_info)


def _actual_sections(value: Any, config: CheckConfig) -> Any:
    if isinstance(value, str):
        return re.sub(r"section0\b", f"section{config.front_section}", value)
    if isinstance(value, dict):
        return {k: _actual_sections(v, config) for k, v in value.items()}
    if isinstance(value, list):
        return [_actual_sections(v, config) for v in value]
    return value


def _diagnostics(context: ReviewContext, rules: Sequence[Rule]) -> dict:
    manuscript, plan, overview = context.manuscript, context.plan, context.overview
    warnings = []
    if plan and not context.plan_info:
        warnings.append("개발계획서에서 이 모듈 코드의 블록을 찾지 못했습니다. ①⑤⑥ 대조는 생략·판단불가입니다")
    elif not plan:
        warnings.append("개발계획서가 없어 ①⑤⑥의 개발계획서 대조는 생략·판단불가입니다")
    elif not context.plan_info["요소"] or not context.plan_info["학습명"]:
        warnings.append("개발계획서 요소 코드 또는 학습명을 읽지 못했습니다")
    for key in ("목표", "선수학습", "내용체계", "핵심 용어"):
        if not overview.get(key):
            warnings.append(f"개요의 '{key}' 블록을 읽지 못했습니다")
    implemented = {rule.item_prefix for rule in rules}
    structure = overview.get("내용체계") or {}
    return {
        "module_code": context.module_code,
        "inputs": {"manuscript": str(manuscript.path), "plan": str(plan.path) if plan else None},
        "format": manuscript.format,
        "sections": {"front": context.config.front_section},
        "counts": {
            "prerequisites": len(overview["선수학습"]["items"]) if overview.get("선수학습") else 0,
            "lessons": len(structure.get("학습", [])),
            "contents": len(structure.get("내용", [])),
            "elements": len(structure.get("요소", [])),
            "terms": len(overview["핵심 용어"]["items"]) if overview.get("핵심 용어") else 0,
        },
        "implemented": sorted(implemented),
        "not_implemented": [item for item in ALL_ITEM_IDS if item not in implemented],
        "warnings": warnings,
    }


def evaluate_review(context: ReviewContext, *, rules: Sequence[Rule] = DEFAULT_RULES) -> ReviewResult:
    config = context.config
    checklist = run_rules(context, rules)
    for row in checklist:
        row["region_path"] = _actual_sections(row["region_path"], config)
        row["비고"] = _actual_sections(row["비고"], config)
    diagnostics = _diagnostics(context, rules)
    return ReviewResult(context.module_code, checklist, flatten(checklist), diagnostics, context)


def run_review(
    paths,
    *,
    roles: dict | None = None,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
    rules: Sequence[Rule] = DEFAULT_RULES,
) -> ReviewResult:
    documents = select_documents(paths, roles, required=("manuscript",))
    manuscript, plan = documents["manuscript"], documents.get("plan")
    context = prepare_review(manuscript, plan, module_code=module_code, config=config)
    return evaluate_review(context, rules=rules)
''')


In [ ]:
# 모듈: ncs_overview/registry.py
write_module('ncs_overview/registry.py', r'''"""규칙 추가 지점. 규칙은 ReviewContext 하나를 받아 기존 형식의 행 목록을 돌려준다."""

from ncs_core import Rule

from .rules.goal import check_goal
from .rules.prerequisites import check_prerequisites
from .rules.structure import check_structure_elements, check_structure_names
from .rules.terms import check_terms

DEFAULT_RULES = (
    Rule("goal", "F02-개요-①", lambda c: check_goal(c.overview, c.plan_info)),
    Rule("prerequisites", "F02-개요-②", lambda c: check_prerequisites(c.overview, c.config.max_prerequisites)),
    Rule("structure_names", "F02-개요-⑤", lambda c: check_structure_names(c.overview, c.plan_info)),
    Rule("structure_elements", "F02-개요-⑥", lambda c: check_structure_elements(c.overview, c.plan_info, c.module_code)),
    Rule("terms", "F02-개요-⑧", lambda c: check_terms(c.overview, c.config.max_terms, c.config.term_max_length)),
)

# 개요 항목 ①~⑧ 중 규칙이 없는 것은 진단의 not_implemented로 보고한다.
# ③ 선수학습 코드 병기, ④ 선수학습 능력단위 수준(NCS 수준 자료 필요), ⑦ 통합 개발 시 요소 전부 기재(통합 사례 필요).
ALL_ITEM_IDS = tuple("F02-개요-" + n for n in "①②③④⑤⑥⑦⑧")
''')


In [ ]:
# 모듈: ncs_overview/reporting.py
write_module('ncs_overview/reporting.py', r'''"""개요 검수 결과 저장. 파일 이름 규칙은 ncs_core.reporting과 같다."""

from ncs_core import ReviewResult, SavedReport, save_report

ITEM_FILES = (
    ("F02-개요-①", "개요①_목표"),
    ("F02-개요-②", "개요②_선수학습"),
    ("F02-개요-⑤", "개요⑤_학습명내용명"),
    ("F02-개요-⑥", "개요⑥_요소코드명칭"),
    ("F02-개요-⑧", "개요⑧_핵심용어"),
)


def save_results(result: ReviewResult, out_dir="결과", *, per_run_dir: bool = True) -> SavedReport:
    return save_report(
        result, out_dir, name="개요", item_files=ITEM_FILES, other_tail="개요_기타", per_run_dir=per_run_dir
    )
''')


In [ ]:
# 모듈: ncs_overview/rules/__init__.py
write_module('ncs_overview/rules/__init__.py', r'''"""문서 읽기와 독립적인 개요 검수 규칙."""
''')


In [ ]:
# 모듈: ncs_overview/rules/goal.py
write_module('ncs_overview/rules/goal.py', r'''"""F02 개요 ①: '학습모듈의 목표' — NCS 능력단위 정의에서 주어를 삭제한 후 '~할 수 있다.'로 종결하였는가.

형식 항목이다. 계획서 문장과 글자가 같은지를 묻지 않으므로
  - 종결어와 주어 삭제는 O/X로 판정하고,
  - 정의와의 관계는 내용이 대응하는지(공백·구두점을 뺀 유사도)만 보아 낮으면 △(사람 확인)로 둔다.
계획서의 '학습모듈의 목표' 줄은 판정에 쓰지 않고 참고로만 적는다.
"""

import difflib
import re

from ncs_core import NA, O, PASS, TRI, X, verdict

from ..patterns import DEFINITION_END, DEFINITION_SUBJECT, GOAL_END

ITEM_ID = "F02-개요-①"
SIMILARITY_MIN = 0.85  # 정의(주어 제거·서술어 변경 후)와 목표의 글자 유사도 하한. 이보다 낮으면 △
SUBJECT_HEAD = re.compile(r"^\S+(이란|란)\s")  # '객실 예약 접수란 ' 꼴 주어가 목표 머리에 남아 있음


def goal_from_definition(definition: str) -> tuple[str, str]:
    """'객실 예약 접수란 … 처리하는 능력이다.' → (주어 '객실 예약 접수란', '… 처리할 수 있다.')"""
    definition = definition.strip()
    m = DEFINITION_SUBJECT.match(definition)
    subject = m.group(0).strip() if m else ""
    body = definition[m.end():] if m else definition
    return subject, DEFINITION_END.sub("할 수 있다.", body).strip()


def _compact(text: str) -> str:
    """공백·구두점·기호 글꼴 가운뎃점을 뺀 글자열 (내용 대응 비교용)."""
    return re.sub(r"[\s·,.\-]", "", text)


def check_goal(overview: dict, plan: dict | None) -> list[dict]:
    goal = overview.get("목표")
    label_end = "① '~할 수 있다.' 종결"
    label_subject = "① 주어 삭제"
    label_def = "① 능력단위 정의 내용과 대응"
    notes: dict[str, str] = {}
    if not goal:
        checks = {label_end: X, label_subject: NA, label_def: NA}
        notes[label_end] = "'학습모듈의 목표' 제목 또는 내용을 찾지 못함"
        return [_row(None, None, checks, notes, "학습모듈의 목표를 작성할 것")]

    text = goal["text"].strip()
    definition = plan.get("정의") if plan else None
    subject, derived = goal_from_definition(definition) if definition else ("", "")

    checks = {label_end: O if GOAL_END.search(text) else X}
    if checks[label_end] == X:
        notes[label_end] = "'~할 수 있다.'로 종결되지 않음"

    # 주어 삭제: 정의의 주어가 목표 머리에 남아 있거나, '…란/이란' 꼴로 시작하면 X
    head = _compact(text)
    if subject and head.startswith(_compact(subject)):
        checks[label_subject] = X
        notes[label_subject] = f'정의의 주어 "{subject}" 가 남아 있음'
    elif SUBJECT_HEAD.match(text):
        checks[label_subject] = X
        notes[label_subject] = f'주어로 보이는 머리 "{SUBJECT_HEAD.match(text).group(0).strip()}" 가 남아 있음'
    else:
        checks[label_subject] = O

    # 정의와의 관계: 글자 일치가 아니라 내용 대응. 낮으면 △
    if definition:
        ratio = difflib.SequenceMatcher(None, _compact(derived), head, autojunk=False).ratio()
        if ratio >= SIMILARITY_MIN:
            checks[label_def] = O
            if _compact(derived) != head:
                notes[label_def] = f'정의와 표현이 조금 다름(유사도 {ratio:.2f}) — 정의 기준 문장: "{derived}"'
        else:
            checks[label_def] = TRI
            notes[label_def] = f'정의와 내용이 다름(유사도 {ratio:.2f}) — 사람 확인. 정의 기준 문장: "{derived}"'
    else:
        checks[label_def] = NA
        notes[label_def] = "개발계획서를 읽지 못해 대조 생략" if not plan else "개발계획서에 능력단위 정의가 없어 대조 생략"

    reference = {"능력단위 정의": definition, "개발계획서 목표(참고)": plan.get("목표")} if plan else None
    result = verdict(checks)
    if result == PASS:
        suggestion = None
    elif checks[label_end] == X or checks[label_subject] == X:
        parts = []
        if checks[label_subject] == X:
            parts.append("주어를 지우고")
        if checks[label_end] == X:
            parts.append("'~할 수 있다.'로 끝맺을 것")
        suggestion = "능력단위 정의에서 " + " ".join(parts)
    else:
        suggestion = "능력단위 정의의 내용과 대응하는지 확인"
    return [_row({"목표": text, "p": goal["p"]}, reference, checks, notes, suggestion)]


def _row(evidence, reference, checks, notes, suggestion) -> dict:
    return {
        "item_id": ITEM_ID,
        "region_path": "section0 > 학습모듈의 개요 > 학습모듈의 목표",
        "원고": evidence,
        "개발계획서": reference,
        "체크": checks,
        "판정": verdict(checks),
        "비고": notes,
        "수정 제안": suggestion,
    }
''')


In [ ]:
# 모듈: ncs_overview/rules/prerequisites.py
write_module('ncs_overview/rules/prerequisites.py', r'''"""F02 개요 ②: '선수학습' 항목 수가 최대 개수(기본 5) 이내인가. '구체적인'은 사람 판단."""

from ncs_core import O, PASS, X, verdict

ITEM_ID = "F02-개요-②"


def check_prerequisites(overview: dict, max_items: int) -> list[dict]:
    block = overview.get("선수학습")
    label = f"② 선수학습 {max_items}개 이내"
    notes: dict[str, str] = {}
    if not block or not block["items"]:
        checks = {label: X}
        notes[label] = "'선수학습' 제목 또는 내용을 찾지 못함"
        evidence = None
    else:
        n = len(block["items"])
        checks = {label: O if n <= max_items else X}
        if n > max_items:
            notes[label] = f"선수학습 {n}개 — {max_items}개 이내로 줄일 것"
        evidence = {"항목": block["items"], "개수": n, "p": block["p"]}
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID,
            "region_path": "section0 > 학습모듈의 개요 > 선수학습",
            "원고": evidence,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else f"구체적인 과목명·학습모듈명(또는 NCS 능력단위)을 {max_items}개 이내로 제시",
        }
    ]
''')


In [ ]:
# 모듈: ncs_overview/rules/structure.py
write_module('ncs_overview/rules/structure.py', r'''"""F02 개요 ⑤·⑥: '학습모듈의 내용 체계' 표의 학습명·학습 내용명(⑤), NCS 능력단위 요소 코드·명칭(⑥)이 개발계획서와 일치하는가."""

from ncs_core import NA, O, PASS, PUA, TRI, X, verdict, visible

from ..compare import diff_list
from ..text import same

ITEM_ID_5 = "F02-개요-⑤"
ITEM_ID_6 = "F02-개요-⑥"
REGION = "section0 > 학습모듈의 개요 > 학습모듈의 내용 체계"


def _missing(item_id: str, label: str, note: str) -> dict:
    return {
        "item_id": item_id,
        "region_path": REGION,
        "원고": None,
        "체크": {label: X},
        "판정": verdict({label: X}),
        "비고": {label: note},
        "수정 제안": "학습모듈의 내용 체계 표를 작성할 것",
    }


def check_structure_names(overview: dict, plan: dict | None) -> list[dict]:
    """⑤ 학습명·학습 내용명 ↔ 개발계획서. 차례 면과의 일치는 차례 검수 ③ 결과를 참조한다."""
    structure = overview.get("내용체계")
    label_l, label_c = "⑤ 학습명 개발계획서 일치", "⑤ 학습 내용명 개발계획서 일치"
    if not structure or not structure["학습"]:
        return [_missing(ITEM_ID_5, label_l, "'학습모듈의 내용 체계' 표 또는 학습명을 읽지 못함")]
    lessons = [visible(f"{L['no']}. {L['text']}") for L in structure["학습"]]
    contents = [visible(f"{s['no']}-{s['sub']}. {s['text']}") for s in structure["내용"]]
    notes: dict = {"근거": structure["근거"]}
    pua = [visible(t) for t in [L["text"] for L in structure["학습"]] + [s["text"] for s in structure["내용"]] if PUA.search(t)]
    if pua:
        # 비교는 기호 글꼴 문자를 가운뎃점으로 치환해서 하고, 문자 자체는 차례 ①에서 X로 보고한다.
        notes["기호 글꼴 문자(참고)"] = "가운뎃점이 기호 글꼴 문자 — ·(U+00B7)로 교체 (차례 ①에서 보고): " + "; ".join(pua)
    if not plan:
        checks = {label_l: TRI, label_c: TRI}
        notes[label_l] = notes[label_c] = "개발계획서를 읽지 못해 대조 불가 — 사람 확인"
    else:
        exp_l = plan["학습명"]
        exp_c = [f"{s['no']}-{s['sub']}. {s['text']}" for s in plan["내용명"]]
        if exp_l:
            value, diff = diff_list("학습명", exp_l, [L["text"] for L in structure["학습"]])
            checks = {label_l: value}
            if diff:
                notes[label_l] = "; ".join(diff)
        else:
            checks = {label_l: TRI}
            notes[label_l] = "개발계획서 학습명을 읽지 못함"
        if exp_c:
            value, diff = diff_list("학습 내용명", exp_c, contents)
            checks[label_c] = value
            if diff:
                notes[label_c] = "; ".join(diff)
        else:
            checks[label_c] = TRI
            notes[label_c] = "개발계획서 학습 내용명을 읽지 못함"
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID_5,
            "region_path": REGION,
            "원고": {"학습": lessons, "학습 내용": contents},
            "개발계획서": {"학습명": plan["학습명"], "내용명": [f"{s['no']}-{s['sub']}. {s['text']}" for s in plan["내용명"]]} if plan else None,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else "개발계획서의 학습명·학습 내용명과 같게 수정 (차례 면도 함께 확인)",
        }
    ]


def check_structure_elements(overview: dict, plan: dict | None, module_code: str) -> list[dict]:
    """⑥ NCS 능력단위 요소 코드 번호·요소 명칭 ↔ 개발계획서."""
    structure = overview.get("내용체계")
    label_set, label_name, label_prefix = "⑥ 요소 코드 번호 개발계획서 일치", "⑥ 요소 명칭 개발계획서 일치", "⑥ 코드 번호가 이 모듈의 능력단위 코드"
    if not structure or not structure["요소"]:
        return [_missing(ITEM_ID_6, label_set, "'학습모듈의 내용 체계' 표에서 요소 코드 번호를 읽지 못함")]
    elements = structure["요소"]
    unit = module_code.replace("LM", "")
    notes: dict = {"근거": structure["근거"]}
    wrong_prefix = [e["code"] for e in elements if not e["code"].startswith(unit + ".")]
    checks = {label_prefix: X if wrong_prefix else O}
    if wrong_prefix:
        notes[label_prefix] = f"능력단위 코드 {unit} 가 아닌 요소 코드: {wrong_prefix}"
    if not plan or not plan["요소"]:
        checks[label_set] = checks[label_name] = TRI
        notes[label_set] = "개발계획서 요소 코드를 읽지 못해 대조 불가 — 사람 확인" if plan else "개발계획서를 읽지 못해 대조 불가 — 사람 확인"
    else:
        exp = {e["code"]: e["name"] for e in plan["요소"]}
        got = {e["code"]: e["name"] for e in elements}
        missing, extra = sorted(exp.keys() - got.keys()), sorted(got.keys() - exp.keys())
        checks[label_set] = O if not missing and not extra else X
        parts = []
        if missing:
            parts.append(f"표에 없는 개발계획서 요소 코드: {missing}")
        if extra:
            parts.append(f"개발계획서에 없는 코드: {extra}")
        if parts:
            notes[label_set] = "; ".join(parts)
        bad = [f'{code}: "{got[code]}" / 기준 "{exp[code]}"' for code in got if code in exp and not same(exp[code], got[code])]
        checks[label_name] = X if bad else (O if got.keys() & exp.keys() else NA)
        if bad:
            notes[label_name] = "요소 명칭 다름 — " + "; ".join(bad)
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID_6,
            "region_path": REGION,
            "원고": [{"코드번호": e["code"], "요소 명칭": visible(e["name"])} for e in elements],
            "개발계획서": plan["요소"] if plan else None,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else "개발계획서의 능력단위 요소 코드 번호·요소명과 같게 수정",
        }
    ]
''')


In [ ]:
# 모듈: ncs_overview/rules/terms.py
write_module('ncs_overview/rules/terms.py', r'''"""F02 개요 ⑧: '핵심 용어'가 최대 개수(기본 10) 이내의 단어(명사형)인가. 명사형 여부는 휴리스틱으로 △만 낸다."""

from ncs_core import O, PASS, TRI, X, verdict

from ..patterns import TERM_PREDICATE

ITEM_ID = "F02-개요-⑧"


def check_terms(overview: dict, max_items: int, max_length: int) -> list[dict]:
    block = overview.get("핵심 용어")
    label_n, label_form = f"⑧ 핵심 용어 {max_items}개 이내", "⑧ 단어(명사형) 형태"
    notes: dict[str, str] = {}
    if not block or not block["items"]:
        checks = {label_n: X}
        notes[label_n] = "'핵심 용어' 제목 또는 내용을 찾지 못함"
        evidence = None
    else:
        items = block["items"]
        n = len(items)
        checks = {label_n: O if n <= max_items else X}
        if n > max_items:
            notes[label_n] = f"핵심 용어 {n}개 — {max_items}개 이내로 줄일 것"
        suspicious = [t for t in items if TERM_PREDICATE.search(t) or len(t) > max_length or t.endswith(".")]
        checks[label_form] = TRI if suspicious else O
        if suspicious:
            notes[label_form] = "단어(명사형)가 아닐 수 있는 항목 — 사람 확인: " + "; ".join(f'"{t}"' for t in suspicious)
        evidence = {"항목": items, "개수": n, "p": block["p"]}
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID,
            "region_path": "section0 > 학습모듈의 개요 > 핵심 용어",
            "원고": evidence,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else f"핵심 용어를 {max_items}개 이내의 단어(명사형)로 제시",
        }
    ]
''')


In [ ]:
# 모듈: ncs_overview/text.py
write_module('ncs_overview/text.py', r'''"""개요 검수의 문자 비교 규칙. 머리 번호 '1. ' / '1-1. '만 지우고 그대로 비교한다.

공백 차이도 다른 글자로 본다: "객실·부대시설" ≠ "객실 · 부대시설", "예약현황" ≠ "예약 현황".
기호 글꼴(PUA) 문자는 가운뎃점으로 바꿔 비교하고(ncs_core.make_normalizer), 그 사실은 비고에 적는다.
"""

from ncs_core import make_normalizer

HEAD_NUMBER = r"^\d+(-\d+)?\.\s*"
norm, same = make_normalizer(HEAD_NUMBER)


def split_items(text: str) -> list[str]:
    """쉼표로 나눈 항목 목록. 괄호 안의 쉼표(예: '체크 인(Check In, 투숙)')는 나누지 않는다."""
    items, depth, current = [], 0, []
    for ch in text:
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth = max(depth - 1, 0)
        if ch in ",，、" and depth == 0:
            items.append("".join(current))
            current = []
        else:
            current.append(ch)
    items.append("".join(current))
    return [item.strip() for item in items if item.strip()]
''')


In [ ]:
# 재실행해도 이전 버전의 import 캐시를 사용하지 않습니다.
for name in list(sys.modules):
    if name.split(".")[0] in ("ncs_core", "ncs_overview"):
        del sys.modules[name]
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))
importlib.invalidate_caches()
from ncs_overview import CheckConfig, run_review, save_results
from ncs_core import select_documents

print("검수 엔진 준비 완료")


## 3. 입력과 설정
`upload`: 원고와 개발계획서를 한 번에 선택합니다(내용으로 역할 판별).
`drive`: Google Drive를 마운트하고 `PATHS`에 경로를 입력합니다. `local`: 로컬 Jupyter 경로를 입력합니다.
파일명에 모듈 코드가 없거나 둘 이상이면 `MODULE_CODE`를 지정하세요. 역할 판별이 안 되면 `ROLES`를 지정하세요.


In [ ]:
INPUT_METHOD = "upload"  # upload / drive / local
PATHS = []  # drive·local: 원고와 개발계획서 경로. upload: 비워 두면 다음 셀에서 선택
ROLES = {}  # 역할 자동 판별이 안 될 때: {"원고.hwpx": "manuscript", "개발계획서.hwpx": "plan"}
MODULE_CODE = ""  # 예: "LM1203020201_24v3". 비우면 원고 파일명에서 추출
CONFIG = CheckConfig(front_section=0, max_prerequisites=5, max_terms=10)
OUTPUT_DIR = "결과"


In [ ]:
result = saved = None
if INPUT_METHOD == "upload":
    from google.colab import files
    upload_root = Path(tempfile.mkdtemp(prefix="ncs_inputs_"))
    print("원고와 개발계획서를 선택하세요 (여러 개 선택 가능).")
    uploaded = files.upload()
    PATHS = []
    for filename, content in uploaded.items():
        path = upload_root / Path(filename).name
        path.write_bytes(content)
        PATHS.append(str(path))
elif INPUT_METHOD == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
elif INPUT_METHOD != "local":
    raise ValueError("INPUT_METHOD는 upload / drive / local 중 하나여야 합니다")
if not PATHS:
    raise ValueError("입력 파일이 없습니다. PATHS를 지정하거나 파일을 업로드하세요")
documents = select_documents(PATHS, ROLES, required=('manuscript',))
for role, document in documents.items():
    print(role, "→", document.path.name, f"({document.format}, 섹션 {len(document.sections)}개)")


## 4. 검사와 추출 결과 확인
먼저 진단의 `warnings`와 `counts`를 확인하세요.


In [ ]:
result = run_review(PATHS, roles=ROLES, module_code=MODULE_CODE, config=CONFIG)
print(json.dumps(result.diagnostics, ensure_ascii=False, indent=2))
print("읽은 개요:")
print(json.dumps(result.context.overview, ensure_ascii=False, indent=2))
print("개발계획서 모듈 블록:")
print(json.dumps(result.context.plan_info, ensure_ascii=False, indent=2))


## 5. 저장과 결과 보기
결과는 실행별 하위 폴더에 저장됩니다.


In [ ]:
if result is None:
    raise ValueError("검사 셀을 먼저 정상 완료하세요")
saved = save_results(result, OUTPUT_DIR)
print(json.dumps(saved.summary, ensure_ascii=False, indent=2))
print(f"지적사항 {len(result.findings)}건")
for finding in result.findings:
    print(finding["item_id"], "|", finding["check"], "|", finding["reason"])
try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    print(json.dumps(result.checklist, ensure_ascii=False, indent=2))
else:
    rows = [{"항목": r["item_id"], "구역": r["region_path"], "판정": r["판정"], "체크": r["체크"], "비고": r["비고"]}
            for r in result.checklist]
    display(pd.DataFrame(rows))
print("결과 ZIP:", saved.zip_path)


In [ ]:
if saved is None:
    raise ValueError("저장 셀을 먼저 정상 완료하세요")
try:
    from google.colab import files
except ImportError:
    print("ZIP 위치:", saved.zip_path)
else:
    files.download(str(saved.zip_path))


## 팀 공유 시 읽는 순서
1. `ncs_overview/pipeline.py`: 검사 전체 흐름과 입력 오류 처리.
2. `ncs_overview/models.py`: 파서와 규칙이 주고받는 데이터.
3. `ncs_overview/parsers.py`: 문서에서 실제로 무엇을 읽었는지.
4. `ncs_overview/rules/`: 판정 기준(goal ①, prerequisites ②, structure ⑤⑥, terms ⑧). `ncs_overview/registry.py`: 새 규칙을 연결할 곳.
5. `ncs_core/`: 두 검수가 공유하는 문서 읽기(hwp/hwpx), 정규화·판정, 입력 역할 판별, 모듈 코드, 저장.

- 개발계획서 없이 실행하면 ①의 대조는 생략(-)되고 ⑤⑥은 △(판단불가)입니다.
- `개요_진단.json`의 `not_implemented`가 이 노트북이 검사하지 않는 항목입니다.
- `*_전체.json`은 모든 결과, `*_지적사항.json`은 X·△만, `*_진단.json`은 입력·읽기 상태를 담습니다.
- 결과는 실행마다 `결과/<모듈코드>_xxxx/` 하위 폴더에 저장되어 이전 결과를 덮어쓰지 않습니다.

페이지 좌표나 OCR을 사용하는 검사가 아니며, 이미지 속 글자를 읽지는 않습니다.
실제 개발계획서·원고 쌍으로 확인한 뒤 운영에 적용하세요.
